In [3]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import pickle

In [4]:
X_train = np.load("X_train_embeddings.npy")
X_test  = np.load("X_test_embeddings.npy")
y_train = np.load("y_train_labels.npy")
y_test  = np.load("y_test_labels.npy")

y_age_train, y_gender_train, y_race_train = y_train[:, 0], y_train[:, 1], y_train[:, 2]
y_age_test,  y_gender_test,  y_race_test  = y_test[:, 0],  y_test[:, 1],  y_test[:, 2]

print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
print(f"Embedding dim: {X_train.shape[1]}")

Train samples: 18964, Test samples: 4741
Embedding dim: 2048


In [5]:
# -- Age Classification --
print("=" * 60)
print("AGE GROUP CLASSIFICATION (XGBClassifier)")
print("=" * 60)

AGE_CLASS_NAMES = ["Young skin (0–18)", "Early aging (19–45)", "Mature skin (46+)"]

def age_to_class(age: int) -> int:
    if age <= 18:
        return 0
    if age <= 45:
        return 1
    return 2

y_age_class_train = np.array([age_to_class(int(a)) for a in y_age_train], dtype=int)
y_age_class_test = np.array([age_to_class(int(a)) for a in y_age_test], dtype=int)

print("Class mapping:")
for i, name in enumerate(AGE_CLASS_NAMES):
    print(f"  Class {i}: {name}")

age_model = XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    max_depth=5,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.1,
    random_state=42,
)
age_model.fit(X_train, y_age_class_train)

y_age_class_pred = age_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_age_class_test, y_age_class_pred):.4f}")
print(classification_report(y_age_class_test, y_age_class_pred, target_names=AGE_CLASS_NAMES))
print("Confusion Matrix:")
print(confusion_matrix(y_age_class_test, y_age_class_pred))

cv_acc = cross_val_score(age_model, X_train, y_age_class_train, cv=5, scoring="accuracy")
print(f"\n5-Fold CV Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})")

AGE GROUP CLASSIFICATION (XGBClassifier)
Class mapping:
  Class 0: Young skin (0–18)
  Class 1: Early aging (19–45)
  Class 2: Mature skin (46+)
Accuracy: 0.8464
                     precision    recall  f1-score   support

  Young skin (0–18)       0.93      0.75      0.83       885
Early aging (19–45)       0.83      0.93      0.88      2747
  Mature skin (46+)       0.84      0.71      0.77      1109

           accuracy                           0.85      4741
          macro avg       0.87      0.80      0.83      4741
       weighted avg       0.85      0.85      0.84      4741

Confusion Matrix:
[[ 662  213   10]
 [  40 2565  142]
 [  11  312  786]]

5-Fold CV Accuracy: 0.8451 (+/- 0.0039)


In [6]:
# -- Gender Classification --
print("\n" + "=" * 60)
print("GENDER CLASSIFICATION (XGBClassifier)")
print("=" * 60)

gender_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    max_depth=4,
    learning_rate=0.1,
    n_estimators=200,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.1,
    random_state=42,
)
gender_model.fit(X_train, y_gender_train)

y_gender_pred = gender_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_gender_test, y_gender_pred):.4f}")
print(classification_report(y_gender_test, y_gender_pred, target_names=["Male", "Female"]))
print("Confusion Matrix:")
print(confusion_matrix(y_gender_test, y_gender_pred))

cv_acc = cross_val_score(gender_model, X_train, y_gender_train, cv=5, scoring="accuracy")
print(f"\n5-Fold CV Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})")


GENDER CLASSIFICATION (XGBClassifier)
Accuracy: 0.8823
              precision    recall  f1-score   support

        Male       0.88      0.90      0.89      2478
      Female       0.89      0.86      0.87      2263

    accuracy                           0.88      4741
   macro avg       0.88      0.88      0.88      4741
weighted avg       0.88      0.88      0.88      4741

Confusion Matrix:
[[2235  243]
 [ 315 1948]]

5-Fold CV Accuracy: 0.8804 (+/- 0.0040)


In [8]:
with open("age_model.pkl", "wb") as f:
    pickle.dump(age_model, f)

with open("gender_model.pkl", "wb") as f:
    pickle.dump(gender_model, f)

print("Saved: age_model.pkl (age-group classifier), gender_model.pkl")

Saved: age_model.pkl (age-group classifier), gender_model.pkl
